# Payer PA Policy Extraction — Hybrid RAG Pipeline

Install runtime dependencies and import all required packages.

In [ ]:
import warnings, sys
warnings.filterwarnings("ignore")

PIP_FLAGS = "--quiet --quiet --no-warn-conflicts --root-user-action=ignore --upgrade"

!pip install {PIP_FLAGS} pypdf>=4.0.0 pymupdf>=1.24 \
    langchain-text-splitters>=0.3.0 \
    langchain-community>=0.3.0 \
    langchain-huggingface>=0.1.0 \
    langchain-openai>=0.2.0 \
    rank-bm25>=0.2.2 \
    chromadb>=0.5.0 \
    sentence-transformers>=2.7.0 \
    pandas>=2.0.0 \
    tqdm>=4.66.0 \
    diskcache>=5.6.3 \
    requests>=2.31.0 \
    rapidfuzz>=3.0.0
print("Dependencies installed.")

In [ ]:
import os, re, json, time, hashlib, sqlite3, logging, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Any
from functools import lru_cache
from datetime import datetime

import requests
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from rapidfuzz import fuzz

from pypdf import PdfReader
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
import chromadb
from chromadb.config import Settings as _ChromaSettings
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

warnings.filterwarnings("ignore")
for _name in ("pydantic", "pkg_resources", "urllib3", "huggingface_hub"):
    logging.getLogger(_name).setLevel(logging.ERROR)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CHROMA_TELEMETRY", "False")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("rag")

Set paths, API credentials, model names, and all pipeline hyperparameters.

In [ ]:
IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working").resolve()
    PDF_DIR      = Path("/kaggle/input/datasets/saurav130/document-pdfs/Sample_PsO_ADS_Track")
else:
    PROJECT_ROOT = Path("/Users/sauravpandey/Downloads/Kaggle/ADS").resolve()
    PDF_DIR      = PROJECT_ROOT / "Sample_PsO_ADS_Track"

CACHE_DIR    = PROJECT_ROOT / ".rag_cache"
OUTPUT_CSV   = PROJECT_ROOT / "submission.csv"
EVAL_DIR     = PROJECT_ROOT / "eval"
CHROMA_DIR   = CACHE_DIR / "chroma"
CACHE_DIR.mkdir(exist_ok=True)
EVAL_DIR.mkdir(exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    OPENROUTER_API_KEY = _secrets.get_secret("OPENROUTER_API_KEY")
    try:
        OPENROUTER_MODEL = _secrets.get_secret("OPENROUTER_MODEL")
    except Exception:
        OPENROUTER_MODEL = "meta-llama/llama-3.1-8b-instruct"
else:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
    OPENROUTER_MODEL   = os.environ.get("OPENROUTER_MODEL", "meta-llama/llama-3.1-8b-instruct")

if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY missing. Set Kaggle secret OPENROUTER_API_KEY or export env var.")

LLM_MODEL       = OPENROUTER_MODEL
LLM_TEMPERATURE = 0.0
LLM_NUM_PREDICT = 2048
API_CALL_DELAY  = 3.0

EMBED_MODEL     = "BAAI/bge-large-en-v1.5"
EMBED_DEVICE    = "cuda" if (not IS_KAGGLE and False) else ("cuda" if IS_KAGGLE else "cpu")

CHUNK_SIZE      = 900
CHUNK_OVERLAP   = 200

TOP_K_BM25      = 12
TOP_K_CHROMA    = 12
RRF_K           = 20
TOP_K_FINAL     = 20

CONFIDENCE_THRESHOLD     = 0.45
FALLBACK_TOKEN           = "Insufficient evidence found"
EVIDENCE_MATCH_THRESHOLD = 70

MAX_PDFS = 70

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"PDFs        : {PDF_DIR}")
print(f"Cache       : {CACHE_DIR}")
print(f"LLM         : {LLM_MODEL} via OpenRouter")
print(f"Embeddings  : {EMBED_MODEL} on {EMBED_DEVICE}")

Verify the OpenRouter API key with a single JSON-mode test request.

In [ ]:
def openrouter_test() -> None:
    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": LLM_MODEL,
            "messages": [{"role": "user", "content": "Return JSON: {\"ok\": true}. No prose."}],
            "response_format": {"type": "json_object"},
            "temperature": 0.0,
            "max_tokens": 64,
        },
        timeout=60,
    )
    r.raise_for_status()
    body = r.json()["choices"][0]["message"]["content"]
    logger.info(f"OpenRouter smoke test OK: {body[:80]}")

openrouter_test()

SQLite-backed LLM cache and JSON-mode call wrapper with retry logic.

In [ ]:
class LLMCache:

    def __init__(self, db_path: Path):
        self.conn = sqlite3.connect(str(db_path), check_same_thread=False)
        self.conn.execute(
            "CREATE TABLE IF NOT EXISTS llm_cache "
            "(key TEXT PRIMARY KEY, response TEXT, created_at REAL)"
        )
        self.conn.commit()

    @staticmethod
    def _key(model: str, prompt: str, system: str) -> str:
        h = hashlib.sha256()
        h.update(model.encode())
        h.update(b"\x00")
        h.update((system or "").encode())
        h.update(b"\x00")
        h.update(prompt.encode())
        return h.hexdigest()

    def get(self, model: str, prompt: str, system: str) -> Optional[dict]:
        row = self.conn.execute(
            "SELECT response FROM llm_cache WHERE key = ?",
            (self._key(model, prompt, system),),
        ).fetchone()
        return json.loads(row[0]) if row else None

    def put(self, model: str, prompt: str, system: str, response: dict) -> None:
        self.conn.execute(
            "INSERT OR REPLACE INTO llm_cache(key, response, created_at) VALUES (?, ?, ?)",
            (self._key(model, prompt, system), json.dumps(response), time.time()),
        )
        self.conn.commit()

LLM_CACHE = LLMCache(CACHE_DIR / "llm.sqlite")

def call_llm_json(
    prompt: str,
    system_prompt: str | None = None,
    *,
    max_tokens: int = LLM_NUM_PREDICT,
    retries: int = 3,
    use_cache: bool = True,
) -> Optional[dict]:
    if use_cache:
        hit = LLM_CACHE.get(LLM_MODEL, prompt, system_prompt or "")
        if hit is not None:
            return hit

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": LLM_MODEL,
        "messages": messages,
        "response_format": {"type": "json_object"},
        "temperature": LLM_TEMPERATURE,
        "max_tokens": max_tokens,
        "top_p": 0.9,
    }
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    raw = ""
    for attempt in range(retries):
        try:
            time.sleep(API_CALL_DELAY)
            r = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers, json=payload, timeout=600,
            )
            r.raise_for_status()
            raw = r.json()["choices"][0]["message"]["content"].strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            parsed = json.loads(raw)
            if use_cache:
                LLM_CACHE.put(LLM_MODEL, prompt, system_prompt or "", parsed)
            return parsed
        except json.JSONDecodeError as e:
            logger.warning(f"[attempt {attempt+1}] JSON parse: {e} | raw[:200]={raw[:200]!r}")
        except Exception as e:
            logger.warning(f"[attempt {attempt+1}] LLM error: {e}")
            time.sleep(max(2 ** attempt, API_CALL_DELAY))
    logger.error("LLM retries exhausted")
    return None

Layout-aware PDF text extraction using PyMuPDF with effective-date detection.

In [ ]:
@dataclass
class PDFDoc:
    filename: str
    pages: list[str]
    full_text: str
    effective_date: Optional[str]
    page_count: int

_DATE_LABEL_RE = re.compile(
    r"(?im)(effective|revision|revised|policy\s+date|approved|last\s+updated|"
    r"reviewed|next\s+review|date\s+of\s+origin|origination|implementation|"
    r"adopted|publish(?:ed)?|date)\s*[:\-]?\s*"
    r"((?:\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})|"
    r"(?:[A-Z][a-z]+\.?\s+\d{1,2},?\s+\d{4})|"
    r"(?:\d{4}-\d{2}-\d{2}))"
)

def _normalize_date(raw: str) -> Optional[str]:
    raw = raw.strip().rstrip(",.;:")
    for fmt in ("%m/%d/%Y", "%m/%d/%y", "%m-%d-%Y", "%m-%d-%y",
                "%B %d, %Y", "%B %d %Y", "%b %d, %Y", "%b %d %Y"):
        try:
            return datetime.strptime(raw, fmt).date().isoformat()
        except ValueError:
            continue
    return None

def _extract_with_pymupdf(pdf_path: Path) -> list[str]:
    pages = []
    doc = fitz.open(str(pdf_path))
    try:
        for page in doc:
            blocks = page.get_text("blocks") or []
            text_blocks = [b for b in blocks if len(b) >= 7 and b[6] == 0 and b[4].strip()]
            if not text_blocks:
                pages.append(page.get_text("text") or "")
                continue
            text_blocks.sort(key=lambda b: (int(b[0] // 250), b[1], b[0]))
            pages.append("\n".join(b[4].strip() for b in text_blocks))
    finally:
        doc.close()
    return pages

def extract_pdf(pdf_path: Path) -> PDFDoc:
    try:
        pages = _extract_with_pymupdf(pdf_path)
    except Exception as e:
        logger.warning(f"PyMuPDF failed on {pdf_path.name}: {e}; falling back to pypdf")
        reader = PdfReader(str(pdf_path))
        pages = [(p.extract_text() or "") for p in reader.pages]
    full_text = "\n\n".join(f"[PAGE {i+1}]\n{t}" for i, t in enumerate(pages))

    window = (pages[0][:1500] if pages else "") + "\n" + (pages[-1][-1500:] if pages else "")
    candidates: list[str] = []
    for _, raw in _DATE_LABEL_RE.findall(window):
        iso = _normalize_date(raw)
        if iso:
            candidates.append(iso)
    effective_date = max(candidates) if candidates else None

    logger.info(
        f"{pdf_path.name}: {len(pages)} pages, "
        f"{len(full_text):,} chars, effective_date={effective_date}"
    )
    return PDFDoc(
        filename=pdf_path.name,
        pages=pages,
        full_text=full_text,
        effective_date=effective_date,
        page_count=len(pages),
    )

def clean_text(raw: str) -> str:
    text = re.sub(r"[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f]", " ", raw)
    for src, tgt in [
        ("\u2019", "'"), ("\u2018", "'"),
        ("\u201c", '"'), ("\u201d", '"'),
        ("\u2013", "-"), ("\u2014", "-"),
        ("\u00ae", ""),  ("\u2022", "-"),
        ("\u2265", ">="), ("\u2264", "<="),
    ]:
        text = text.replace(src, tgt)
    text = re.sub(
        r"(?im)^(page\s+\d+(\s+of\s+\d+)?|confidential|proprietary|"
        r"copyright|all rights reserved)\s*$",
        "", text,
    )
    text = re.sub(r"\n{3,}", "\n\n", text)
    lines = [re.sub(r"[ \t]{2,}", " ", ln).strip() for ln in text.splitlines()]
    return "\n".join(lines).strip()

Split cleaned text into overlapping chunks tagged with page and position metadata.

In [ ]:
@dataclass
class Chunk:
    idx: int
    text: str
    page: int
    rel_position: float

    def __hash__(self) -> int:
        return hash((self.idx, self.page))

_PAGE_HEAD_RE = re.compile(r"\[PAGE (\d+)\]")

def chunk_document(cleaned: str, page_count: int) -> list[Chunk]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", ", ", " "],
    )
    raw_chunks = splitter.split_text(cleaned)
    chunks: list[Chunk] = []
    total = len(cleaned) or 1
    running_offset = 0
    current_page = 1
    for i, txt in enumerate(raw_chunks):
        m = _PAGE_HEAD_RE.search(txt)
        if m:
            current_page = int(m.group(1))
        running_offset = cleaned.find(txt, running_offset)
        rel = (running_offset / total) if running_offset >= 0 else (i / max(len(raw_chunks), 1))
        chunks.append(Chunk(idx=i, text=txt, page=current_page, rel_position=rel))
        running_offset = max(running_offset, 0) + len(txt)
    logger.info(f"Chunked into {len(chunks)} chunks (avg page={np.mean([c.page for c in chunks]):.1f})")
    return chunks

Load the BGE-large embedding model and build a per-document ChromaDB vector index.

In [ ]:
logger.info(f"Loading embedding model {EMBED_MODEL} on {EMBED_DEVICE}...")
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": EMBED_DEVICE},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)
logger.info("Embedding model loaded.")

_TOKEN_RE = re.compile(r"[a-z0-9]+")
def _tokenize(text: str) -> list[str]:
    return _TOKEN_RE.findall(text.lower())

def build_bm25_index(chunks: list[Chunk]) -> BM25Okapi:
    tokenized = [_tokenize(c.text) for c in chunks]
    return BM25Okapi(tokenized)

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=_ChromaSettings(anonymized_telemetry=False),
)

def build_chroma_collection(chunks: list[Chunk], doc_signature: str):
    name = f"doc_{doc_signature}"
    try:
        chroma_client.delete_collection(name=name)
    except Exception:
        pass
    coll = chroma_client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    texts = [c.text for c in chunks]
    embeddings = embedding_model.embed_documents(texts)
    coll.add(
        ids=[str(c.idx) for c in chunks],
        documents=texts,
        embeddings=embeddings,
        metadatas=[
            {"idx": c.idx, "page": c.page, "rel_position": c.rel_position}
            for c in chunks
        ],
    )
    return coll

BM25 lexical and ChromaDB semantic retrieval fused via Reciprocal Rank Fusion.

In [ ]:
@dataclass
class RetrievalResult:
    chunks: list[Chunk]
    bm25_ranks: dict[int, int]
    chroma_ranks: dict[int, int]
    rrf_scores: dict[int, float]

    @property
    def faiss_ranks(self) -> dict[int, int]:
        return self.chroma_ranks

def reciprocal_rank_fusion(
    bm25_order: list[int],
    semantic_order: list[int],
    k: int = RRF_K,
) -> dict[int, float]:
    scores: dict[int, float] = {}
    for rank, idx in enumerate(bm25_order, start=1):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    for rank, idx in enumerate(semantic_order, start=1):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    return scores

def hybrid_retrieve(
    query: str,
    chunks: list[Chunk],
    bm25: BM25Okapi,
    chroma_collection,
    top_k_final: int = TOP_K_FINAL,
) -> RetrievalResult:
    scores = bm25.get_scores(_tokenize(query))
    bm25_order = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)[:TOP_K_BM25]
    bm25_ranks = {idx: r + 1 for r, idx in enumerate(bm25_order)}

    q_emb = embedding_model.embed_query(query)
    res = chroma_collection.query(
        query_embeddings=[q_emb],
        n_results=min(TOP_K_CHROMA, len(chunks)),
        include=["metadatas"],
    )
    chroma_order = [int(m["idx"]) for m in (res["metadatas"][0] if res["metadatas"] else [])]
    chroma_ranks = {idx: r + 1 for r, idx in enumerate(chroma_order)}

    rrf = reciprocal_rank_fusion(bm25_order, chroma_order)
    fused_order = sorted(rrf.keys(), key=lambda i: rrf[i], reverse=True)[:top_k_final]
    fused_chunks = [chunks[i] for i in fused_order]

    return RetrievalResult(
        chunks=fused_chunks,
        bm25_ranks=bm25_ranks,
        chroma_ranks=chroma_ranks,
        rrf_scores=rrf,
    )

Freshness, trust, and retrieval-consistency signals aggregated into source confidence.

In [ ]:
def compute_freshness_score(effective_date: Optional[str], today: Optional[datetime] = None) -> float:
    if not effective_date:
        return 0.7
    try:
        d = datetime.fromisoformat(effective_date)
    except ValueError:
        return 0.7
    today = today or datetime.now()
    years = max(0.0, (today - d).days / 365.25)
    return float(max(0.05, min(1.0, np.exp(-years / 3.0))))

def compute_trust_score(result: RetrievalResult, page_count: int) -> float:
    if not result.chunks:
        return 0.0
    positions = np.array([c.rel_position for c in result.chunks])
    body_mass = float(np.mean((positions > 0.05) & (positions < 0.85)))
    page_spread = len({c.page for c in result.chunks}) / max(1, min(page_count, len(result.chunks)))
    return float(0.6 * body_mass + 0.4 * min(1.0, page_spread))

def compute_retrieval_consistency(result: RetrievalResult, top_k: int = 10) -> float:
    bm25_top = {i for i, r in result.bm25_ranks.items() if r <= top_k}
    faiss_top = {i for i, r in result.faiss_ranks.items() if r <= top_k}
    if not bm25_top or not faiss_top:
        return 0.0
    inter = bm25_top & faiss_top
    union = bm25_top | faiss_top
    jacc = len(inter) / len(union)
    appears_in_both = sum(
        1 for c in result.chunks
        if c.idx in result.bm25_ranks and c.idx in result.faiss_ranks
    )
    coverage = appears_in_both / max(1, len(result.chunks))
    return float(0.5 * jacc + 0.5 * coverage)

def aggregate_source_confidence(freshness: float, trust: float, consistency: float) -> float:
    eps = 1e-3
    f = max(freshness, 0.5)
    t = max(trust, 0.3)
    c = max(consistency, 0.3)
    return float(
        np.exp(
            0.10 * np.log(f + eps)
            + 0.45 * np.log(t + eps)
            + 0.45 * np.log(c + eps)
        )
    )

Disk-backed retrieval cache keyed by document hash and normalized query string.

In [ ]:
import diskcache

RETRIEVAL_CACHE = diskcache.Cache(str(CACHE_DIR / "retrieval"))

def _doc_hash(text: str) -> str:
    return hashlib.sha1(text.encode()).hexdigest()[:16]

@lru_cache(maxsize=2048)
def _normalize_query(q: str) -> str:
    return " ".join(_tokenize(q))

def cached_hybrid_retrieve(
    query: str,
    doc_signature: str,
    chunks: list[Chunk],
    bm25: BM25Okapi,
    chroma_collection,
) -> RetrievalResult:
    key = (doc_signature, _normalize_query(query))
    hit = RETRIEVAL_CACHE.get(key)
    if hit is not None:
        return RetrievalResult(
            chunks=[Chunk(**c) for c in hit["chunks"]],
            bm25_ranks=hit["bm25_ranks"],
            chroma_ranks=hit["chroma_ranks"],
            rrf_scores=hit["rrf_scores"],
        )
    result = hybrid_retrieve(query, chunks, bm25, chroma_collection)
    RETRIEVAL_CACHE.set(key, {
        "chunks": [asdict(c) for c in result.chunks],
        "bm25_ranks": result.bm25_ranks,
        "chroma_ranks": result.chroma_ranks,
        "rrf_scores": result.rrf_scores,
    })
    return result

Detect TREMFYA and STELARA via regex scan then verify indication with a targeted LLM call.

In [ ]:
TARGET_BRANDS: set[str] = {"TREMFYA", "STELARA"}

BRAND_ALIASES: dict[str, list[str]] = {
    "TREMFYA": ["TREMFYA", "GUSELKUMAB"],
    "STELARA": ["STELARA", "USTEKINUMAB"],
}
BRAND_GENERIC: dict[str, str] = {
    "TREMFYA": "guselkumab",
    "STELARA": "ustekinumab",
}

BRAND_DETECTION_SYSTEM = (
    "You are a medical policy parser. Use ONLY the provided text. "
    "Do NOT use external knowledge. Output strict JSON only."
)

INDICATION_CHECK_PROMPT = """Does the PA policy text below have its OWN approval criteria
(not just mentioned as a step-therapy alternative) for {brand} ({generic})
under the PSORIASIS (PsO) indication?

Reply with ONLY: {{"pso": true}} or {{"pso": false}}

TEXT:
{text}"""

PSA_CHECK_PROMPT = """Does the PA policy text below have its OWN approval criteria
(not just mentioned as a step-therapy alternative) for {brand} ({generic})
under the PSORIATIC ARTHRITIS (PsA) indication?

Reply with ONLY: {{"psa": true}} or {{"psa": false}}

TEXT:
{text}"""

def _brands_in_text(text: str) -> set[str]:
    text_upper = text.upper()
    return {brand for brand, aliases in BRAND_ALIASES.items()
            if any(alias in text_upper for alias in aliases)}

def detect_brands(cleaned: str) -> list[str]:
    present = _brands_in_text(cleaned)
    if not present:
        logger.info("No target brands (TREMFYA / STELARA) found in document text")
        return []

    logger.info(f"Brands found in text: {sorted(present)}")

    sample = cleaned[:15000]

    by_ind: dict[str, set[str]] = {"PSO": set(), "PSA": set()}

    for brand in sorted(present):
        generic = BRAND_GENERIC[brand]

        result = call_llm_json(
            INDICATION_CHECK_PROMPT.format(brand=brand, generic=generic, text=sample),
            system_prompt=BRAND_DETECTION_SYSTEM,
            max_tokens=32,
        )
        if result is not None:
            if result.get("pso"):
                by_ind["PSO"].add(brand)
                continue

        result_psa = call_llm_json(
            PSA_CHECK_PROMPT.format(brand=brand, generic=generic, text=sample),
            system_prompt=BRAND_DETECTION_SYSTEM,
            max_tokens=32,
        )
        if result_psa is not None and result_psa.get("psa"):
            by_ind["PSA"].add(brand)
            continue

        brand_pos = cleaned.upper().find(brand)
        if brand_pos >= 0:
            window = cleaned[max(0, brand_pos - 300): brand_pos + 2000].upper()
            if any(kw in window for kw in ["PSORIASIS", "PLAQUE", " PSO", "PSOR"]):
                logger.warning(f"{brand}: LLM indication check failed, regex → PsO")
                by_ind["PSO"].add(brand)
            else:
                logger.warning(f"{brand}: LLM indication check failed, regex → PsA fallback")
                by_ind["PSA"].add(brand)

    pso = sorted(by_ind["PSO"])
    if pso:
        logger.info(f"PsO brands detected: {pso}")
        return pso
    psa = sorted(by_ind["PSA"])
    if psa:
        logger.info(f"PsO not found; falling back to PsA brands: {psa}")
        return psa
    fallback = sorted(present)
    logger.warning(f"Indication checks inconclusive; returning all found brands: {fallback}")
    return fallback

Query strings and extraction definitions for all 12 PA business parameters.

In [ ]:
PARAMETERS: dict[str, dict[str, str]] = {
    "Age": {
        "query": "{brand} age eligibility minimum age requirement years old pediatric adult adolescent threshold FDA labelled",
        "definition": (
            "Age eligibility criteria for {brand} for the Psoriasis (PsO) indication. "
            "If the policy does NOT specify a numerical threshold but refers to what the drug "
            "is FDA-indicated for, output 'FDA labelled age'. "
            "OUTPUT FORMAT: output as an inequality string e.g. '>=18', '>6 years', '>=12', "
            "or the fixed phrase 'FDA labelled age'. "
            "If two age groups are listed, capture the YOUNGEST. "
            "Output NOT_FOUND if age criteria are not mentioned at all."
        ),
        "null_default": "Not specified",
    },
    "Step_Therapy_Requirements": {
        "query": "{brand} step therapy prior failure trial inadequate response intolerance contraindication conventional systemic biologic universal criteria all indications",
        "definition": (
            "Extract ALL step therapy language from the policy that applies to {brand} for "
            "Psoriasis (PsO). Include BOTH: (1) brand-specific or indication-specific step "
            "requirements, AND (2) universal/general criteria that apply across all indications "
            "in the policy. Include phototherapy/PUVA language when it appears within step "
            "statements. If the policy distinguishes between moderate-to-severe and severe PsO, "
            "capture ONLY the moderate-to-severe criteria. Return verbatim text from the policy. "
            "Output NOT_FOUND if no step therapy criteria are present."
        ),
        "null_default": "None required",
    },
    "Number_of_Steps_through_Brands": {
        "query": "{brand} branded biologic biosimilar TNF IL-17 IL-23 preferred adalimumab ustekinumab etanercept secukinumab ixekizumab tried failed step prior drug class",
        "definition": (
            "Count the number of BRANDED or BIOLOGIC step therapy steps required before {brand} "
            "can be approved for PsO. Rules:"
            "1. A preferred ustekinumab or adalimumab product counts as a branded step."
            "2. If the policy references a drug class (e.g., TNF inhibitors, IL-17 antagonists) "
            "   and {brand} belongs to that class, the class-level step counts as a branded step."
            "3. Take the UNION of universal criteria steps AND indication/brand-specific steps — "
            "   these are joined by AND, meaning both sets must be satisfied."
            "4. If any steps appear in an OR statement, take the path with FEWER steps."
            "5. If the policy distinguishes moderate-to-severe vs. severe PsO, use only "
            "   moderate-to-severe criteria."
            "6. Do NOT count phototherapy steps here."
            "Output a single number (e.g., 1, 2) or 'NA' if no branded steps are required."
        ),
        "null_default": "NA",
    },
    "Number_of_Steps_through_Generic": {
        "query": "{brand} topical corticosteroid methotrexate cyclosporine acitretin retinoid apremilast non-biologic conventional systemic generic step prior",
        "definition": (
            "Count the number of GENERIC or NON-BIOLOGIC step therapy steps required before "
            "{brand} can be approved for PsO. Rules:"
            "1. Topical agents (corticosteroids, vitamin D analogues, etc.) count as generic steps."
            "2. If a parent indication requires a step but does not name any brand or biologic, "
            "   it defaults to a generic step."
            "3. If preferred and non-preferred products are listed and {brand} is non-preferred, "
            "   any universal criteria requiring steps through preferred alternatives also count — "
            "   steps that explicitly mention biologics count as branded; all others count as generic."
            "4. Take the UNION of universal criteria steps AND indication/brand-specific steps "
            "   (AND logic). For OR statements, take the path with FEWER steps."
            "5. If the policy distinguishes moderate-to-severe vs. severe PsO, use only "
            "   moderate-to-severe criteria."
            "6. Do NOT count phototherapy steps here."
            "Output a single number (e.g., 1, 2) or 'NA' if no generic steps are required."
        ),
        "null_default": "NA",
    },
    "Step_Through_Phototherapy": {
        "query": "{brand} phototherapy PUVA UVB narrow-band light therapy psoralen ultraviolet step prior failure required mandatory",
        "definition": (
            "Does the policy require stepping through phototherapy (including PUVA — psoralen "
            "combined with UVA light exposure) before {brand} can be approved for PsO? "
            "Apply the same union logic as step therapy: combine universal criteria phototherapy "
            "language with indication/brand-specific phototherapy language (AND logic), then assess:"
            "  'Yes' — phototherapy is a mandatory required step and is NOT in an OR statement"
            "  'No'  — policy does not mention phototherapy as a required step for drug approval"
            "  'N/A' — policy lists no step therapy criteria at all"
        ),
        "null_default": "No",
    },
    "TB_Test_Required": {
        "query": "{brand} tuberculosis TB test required screening baseline PPD tuberculin skin test TST interferon-gamma release assay IGRA QuantiFERON-TB Gold latent active TB prior to initiating pre-treatment safety screening",
        "definition": (
            "Does the policy require a Tuberculosis (TB) test before {brand} can be approved "
            "for PsO? TB testing may be referred to as: TB test, PPD (purified protein derivative), "
            "tuberculin skin test (TST), IGRA (interferon-gamma release assay), QuantiFERON-TB Gold, "
            "or baseline/pre-treatment screening for latent or active TB."
            "  'Y'             — TB test is explicitly required before initiation"
            "  'N'             — policy explicitly states TB test is not required"
            "  'Not specified' — policy does not mention TB testing at all"
        ),
        "null_default": "Not specified",
    },
    "Initial_Authorization_Duration": {
        "query": "{brand} initial authorization duration period months coverage approved first PA approval length granted",
        "definition": (
            "The time period (in months) for which initial PA coverage is granted for {brand} "
            "for PsO. This is the first-approval duration only, not reauthorization. "
            "If PA is indicated for PsO (i.e., PA is required), output must be either the "
            "numeric duration in months (e.g., 12, 6) or 'Unspecified' if PA is required but "
            "the duration is not stated. Include the unit in the output — output '6 Months', '12 Months', "
            "not just the number. 'Unspecified' if duration not stated."
        ),
        "null_default": "Unspecified",
    },
    "Reauthorization_Duration": {
        "query": "{brand} reauthorization renewal continuation re-approval subsequent months duration after initial period reassessment",
        "definition": (
            "The length of time (in months) for which reauthorization is granted once the initial "
            "approval period ends for {brand} for PsO. Often expressed in 6- or 12-month intervals. "
            "If 'Reauthorization Required' is Yes, output must be either the numeric duration "
            "in months (e.g., 12, 6) or 'Unspecified' if reauth is required but duration is "
            "not stated. Output 'NA' if reauthorization is explicitly not required. "
            "Include the unit — output '12 Months', '6 Months', not just the number."
        ),
        "null_default": "Unspecified",
    },
    "Reauthorization_Required": {
        "query": "{brand} reauthorization required renewal continued coverage reassessment documentation after initial authorization",
        "definition": (
            "Is reauthorization required for continued coverage of {brand} for PsO after the "
            "initial authorization period expires? IMPORTANT RULE: if either Reauthorization "
            "Duration OR Reauthorization Requirements are present (non-NA/non-empty) in the policy, "
            "this field MUST be 'Yes'. If the policy explicitly states no reauthorization is "
            "needed, output 'No'. Output 'Not specified' only if no reauth language of any kind "
            "is present."
            "  'Yes' — reauth is required (or duration/criteria are documented)"
            "  'No'  — policy explicitly states reauth is not needed"
            "  'Not specified' — no reauth language of any kind found"
        ),
        "null_default": "Not specified",
    },
    "Reauthorization_Requirements": {
        "query": "{brand} reauthorization continued clinical benefit response improvement PASI BSA stable maintained therapy continuation criteria documentation lab values physician attestation",
        "definition": (
            "Explicit reauthorization or continuation criteria that must be met for {brand} "
            "for PsO to receive renewed coverage. This includes: continued clinical benefit, "
            "documented response to therapy (e.g., PASI improvement, BSA reduction), absence "
            "of disease progression, specific lab values, physician attestation, or other "
            "clinical documentation requirements. Return verbatim text from the policy. "
            "Output NOT_FOUND if the policy does not specify any continuation criteria."
        ),
        "null_default": "Not specified",
    },
    "Specialist_Types": {
        "query": "{brand} prescriber specialist dermatologist rheumatologist gastroenterologist physician prescribed by managed by initiating specialist restriction required",
        "definition": (
            "Identifies the specific medical specialties that are acceptable for initiating or "
            "managing treatment with {brand} for PsO. Some policies restrict prescribing to "
            "specific specialists (e.g., dermatologists, rheumatologists, gastroenterologists). "
            "List the specialist types mentioned. Output NOT_FOUND if no specialist restriction "
            "is mentioned in the policy."
        ),
        "null_default": "No restriction specified",
    },
    "Quantity_Limits": {
        "query": "{brand} quantity limit units per month supply allowable maximum vials syringes pens dispense",
        "definition": (
            "Quantity limits for {brand} for PsO — ONLY capture what is explicitly labeled "
            "'quantity limit' or 'quantity limitation' in the policy. "
            "IMPORTANT: Do NOT capture text that is labeled as 'dosage', 'dosing', 'dose', "
            "or 'dosing limit' — those are clinical dosing instructions, not quantity limits. "
            "Only policy-level quantity restrictions count. Output NOT_FOUND if not explicitly stated."
        ),
        "null_default": "Not specified",
    },
}

Build retrieval context per brand, call the LLM, and apply confidence and evidence gates.

In [ ]:
EXTRACTION_SYSTEM = """You are a medical policy extraction engine.
STRICT RULES:
1. Extract information ONLY from the document context provided.
2. NEVER use your own training knowledge, medical background, or guesses.
3. If the information is not explicitly present in the context, return "NOT_FOUND".
4. Do NOT infer, assume, or extrapolate.
5. Respond with ONLY a valid JSON object. No markdown, no prose.
6. Evidence MUST be a direct phrase copy-pasted from the document context, under 30 words.
"""

EXTRACTION_PROMPT = """DOCUMENT CONTEXT (retrieved from payer PA policy):
=====================================================
{context}
=====================================================

Drug Brand: {brand}

For each parameter below, extract the value from the context above ONLY. If a parameter
is not explicitly present in the context, set value to "NOT_FOUND" with confidence 0.0
and evidence null.

PARAMETERS:
{param_block}

Respond with EXACTLY this JSON shape:
{{
  "<Parameter_Name>": {{
    "value": "<extracted value or NOT_FOUND>",
    "confidence": <float 0.0-1.0>,
    "evidence": "<verbatim phrase from document or null>"
  }},
  ...
}}

Confidence: 1.0 explicit, 0.7 implied, 0.4 ambiguous, 0.0 not found.
"""

def build_retrieval_context_for_brand(
    brand: str,
    chunks: list[Chunk],
    bm25: BM25Okapi,
    chroma_collection,
    doc_signature: str,
    per_param_chunks: int = 5,
    max_chunks_total: int = 36,
) -> tuple[str, dict[str, RetrievalResult]]:
    seen: set[int] = set()
    merged: list[Chunk] = []
    per_param: dict[str, RetrievalResult] = {}

    for param_name, cfg in PARAMETERS.items():
        query = cfg["query"].format(brand=brand)
        result = cached_hybrid_retrieve(query, doc_signature, chunks, bm25, chroma_collection)
        per_param[param_name] = result
        for c in result.chunks[:per_param_chunks]:
            if c.idx in seen:
                continue
            seen.add(c.idx)
            merged.append(c)
            if len(merged) >= max_chunks_total:
                break
        if len(merged) >= max_chunks_total:
            break

    context = "\n\n---\n\n".join(
        f"[chunk {c.idx} | page {c.page}]\n{c.text}" for c in merged
    )
    return context, per_param

def evidence_in_context(evidence: str, context: str, threshold: int = EVIDENCE_MATCH_THRESHOLD) -> bool:
    if not evidence or not evidence.strip():
        return False
    score = fuzz.partial_ratio(evidence.lower(), context.lower())
    return score >= threshold

def extract_all_parameters(
    brand: str,
    chunks: list[Chunk],
    bm25: BM25Okapi,
    chroma_collection,
    doc_signature: str,
    source_confidence: float,
) -> dict[str, dict]:
    context, _per_param = build_retrieval_context_for_brand(
        brand, chunks, bm25, chroma_collection, doc_signature,
    )

    param_lines = "\n".join(
        f"- {name}: {cfg['definition'].format(brand=brand)}"
        for name, cfg in PARAMETERS.items()
    )
    prompt = EXTRACTION_PROMPT.format(context=context, brand=brand, param_block=param_lines)
    raw = call_llm_json(prompt, system_prompt=EXTRACTION_SYSTEM, max_tokens=LLM_NUM_PREDICT) or {}

    out: dict[str, dict] = {}
    for param_name in PARAMETERS:
        entry = raw.get(param_name) or {}
        value = str(entry.get("value", "NOT_FOUND")).strip()
        conf  = float(entry.get("confidence", 0.0) or 0.0)
        evid  = entry.get("evidence")
        evid_str = str(evid).strip() if evid else ""

        verified = evidence_in_context(evid_str, context) if evid_str else False

        effective_conf = conf * (0.5 + 0.5 * source_confidence)
        if value == "NOT_FOUND":
            value_out = PARAMETERS[param_name]["null_default"]
        elif conf < CONFIDENCE_THRESHOLD:
            value_out = FALLBACK_TOKEN
        elif evid_str and not verified:
            value_out = FALLBACK_TOKEN
        else:
            value_out = value

        out[param_name] = {
            "value": value_out,
            "raw_value": value,
            "confidence": conf,
            "effective_confidence": effective_conf,
            "evidence": evid_str,
            "evidence_verified": verified,
        }
    return out

Reconcile reauthorization fields, normalize auth duration, and clean quantity limits.

In [ ]:
def post_process_reauthorization(row: dict) -> dict:
    dur  = str(row.get("Reauthorization_Duration", ""))
    reqs = str(row.get("Reauthorization_Requirements", ""))
    req  = str(row.get("Reauthorization_Required", ""))
    placeholder = {"NA", "Not specified", "Unspecified", "NOT_FOUND", "", FALLBACK_TOKEN, "None required"}

    dur_is_real  = dur not in placeholder
    reqs_is_real = reqs not in placeholder

    if dur_is_real or reqs_is_real:
        row["Reauthorization_Required"] = "Yes"
    elif req.strip().lower() == "yes":
        row["Reauthorization_Required"] = FALLBACK_TOKEN
    return row

def post_process_initial_auth(row: dict) -> dict:
    val = row.get("Initial_Authorization_Duration", "")
    if val in {"Not specified", "NOT_FOUND", "", FALLBACK_TOKEN}:
        row["Initial_Authorization_Duration"] = "Unspecified"
    return row

def post_process_quantity_limits(row: dict) -> dict:
    val = str(row.get("Quantity_Limits", ""))
    if re.search(r"\b(dosage|dosing limit|dose)\b", val, re.IGNORECASE):
        row["Quantity_Limits"] = "Not specified"
    return row

def apply_business_rules(row: dict) -> dict:
    row = post_process_reauthorization(row)
    row = post_process_initial_auth(row)
    row = post_process_quantity_limits(row)
    return row

Compute Access Score (0–100) as a weighted sum of seven field-level scorers.

In [ ]:
def _score_age(val: str) -> float:
    v = str(val).lower()
    if "insufficient evidence" in v: return 0.0
    if "not specified" in v: return 0.0
    if "fda labelled" in v: return 12.0
    m = re.search(r"(\d+)", v)
    if not m:
        return 10.0
    age = int(m.group(1))
    if age <= 18: return 15.0
    if age <= 21: return 13.0
    if age <= 30: return 10.0
    if age <= 50: return 6.0
    return 3.0

def _score_steps(brand_v: str, generic_v: str) -> float:
    sb = str(brand_v).strip().upper()
    sg = str(generic_v).strip().upper()
    if "INSUFFICIENT" in sb and "INSUFFICIENT" in sg:
        return 0.0
    def parse(x):
        s = str(x).strip().upper()
        if s in ("NA", "INSUFFICIENT EVIDENCE FOUND"):
            return 0
        m = re.search(r"\d+", s)
        return int(m.group()) if m else 0
    total = parse(brand_v) + parse(generic_v)
    return {0: 35.0, 1: 28.0, 2: 20.0, 3: 12.0, 4: 7.0}.get(total, 3.0)

def _score_phototherapy(val: str) -> float:
    v = str(val).strip().lower()
    if v == "yes": return 0.0
    if "insufficient evidence" in v: return 0.0
    return 10.0

def _score_auth_duration(val: str) -> float:
    v = str(val).strip().lower()
    if "insufficient evidence" in v: return 0.0
    if v in ("unspecified", "not specified"):
        return 6.0
    m = re.search(r"(\d+)", v)
    if not m:
        return 10.0
    months = int(m.group(1))
    if months >= 12: return 15.0
    if months >= 6:  return 8.0
    return 4.0

def _score_tb(val: str) -> float:
    v = str(val).strip().upper()
    if "INSUFFICIENT" in v: return 0.0
    if v.startswith("Y"): return 0.0
    if v.startswith("N"): return 5.0
    return 2.0

def _score_reauthorization(required: str, duration: str) -> float:
    if str(required).strip().lower() == "no":
        return 10.0
    m = re.search(r"(\d+)", str(duration))
    if not m:
        return 5.0
    months = int(m.group(1))
    if months >= 12: return 7.0
    if months >= 6:  return 4.0
    return 2.0

def _score_specialist(val: str) -> float:
    s = str(val).lower()
    if "insufficient evidence" in s: return 0.0
    if any(x in s for x in ("no restriction", "not specified")):
        return 10.0
    return 5.0

def compute_access_score(row: dict) -> float:
    score = 0.0
    score += _score_age(row.get("Age", "Not specified"))
    score += _score_steps(
        row.get("Number_of_Steps_through_Brands", "NA"),
        row.get("Number_of_Steps_through_Generic", "NA"),
    )
    score += _score_phototherapy(row.get("Step_Through_Phototherapy", "No"))
    score += _score_auth_duration(row.get("Initial_Authorization_Duration", "Unspecified"))
    score += _score_tb(row.get("TB_Test_Required", "Not specified"))
    score += _score_reauthorization(
        row.get("Reauthorization_Required", "Not specified"),
        row.get("Reauthorization_Duration", "Unspecified"),
    )
    score += _score_specialist(row.get("Specialist_Types", "No restriction specified"))
    return round(min(score, 100.0), 1)

Sequential per-PDF processing with incremental CSV writes and crash-safe resume.

In [ ]:
SUBMISSION_COLUMN_ORDER = [
    "Filename", "Brand", "Age", "Step_Therapy_Requirements",
    "Number_of_Steps_through_Brands", "Number_of_Steps_through_Generic",
    "Step_Through_Phototherapy", "TB_Test_Required",
    "Initial_Authorization_Duration", "Reauthorization_Duration",
    "Reauthorization_Required", "Reauthorization_Requirements",
    "Specialist_Types", "Quantity_Limits", "Access_Score",
]
DIAGNOSTIC_COLUMNS = [
    "Freshness_Score", "Trust_Score", "Retrieval_Consistency",
    "Source_Confidence", "Mean_Field_Confidence", "Fallback_Rate",
]
COLUMN_ORDER = SUBMISSION_COLUMN_ORDER + DIAGNOSTIC_COLUMNS

def process_pdf(pdf_path: Path) -> list[dict]:
    doc = extract_pdf(pdf_path)
    cleaned = clean_text(doc.full_text)
    if not cleaned:
        return []

    brands = detect_brands(cleaned)
    if not brands:
        logger.info(f"  no brands found in {doc.filename}, skipping")
        return []

    chunks = chunk_document(cleaned, doc.page_count)
    if not chunks:
        return []

    doc_sig = _doc_hash(cleaned)
    bm25 = build_bm25_index(chunks)
    chroma_collection = build_chroma_collection(chunks, doc_sig)

    freshness = compute_freshness_score(doc.effective_date)

    rows = []
    for brand in brands:
        logger.info(f"  -> extracting {brand}")
        _ctx, per_param = build_retrieval_context_for_brand(
            brand, chunks, bm25, chroma_collection, doc_sig,
        )
        trust = float(np.mean([
            compute_trust_score(r, doc.page_count) for r in per_param.values()
        ]))
        consistency = float(np.mean([
            compute_retrieval_consistency(r) for r in per_param.values()
        ]))
        source_conf = aggregate_source_confidence(freshness, trust, consistency)

        extracted = extract_all_parameters(
            brand, chunks, bm25, chroma_collection, doc_sig, source_conf,
        )

        row = {"Filename": doc.filename, "Brand": brand}
        confidences = []
        fallback_count = 0
        for param_name, entry in extracted.items():
            row[param_name] = entry["value"]
            confidences.append(entry["effective_confidence"])
            if entry["value"] == FALLBACK_TOKEN:
                fallback_count += 1

        row = apply_business_rules(row)
        row["Access_Score"] = compute_access_score(row)
        row["Freshness_Score"]        = round(freshness, 3)
        row["Trust_Score"]            = round(trust, 3)
        row["Retrieval_Consistency"]  = round(consistency, 3)
        row["Source_Confidence"]      = round(source_conf, 3)
        row["Mean_Field_Confidence"]  = round(float(np.mean(confidences)) if confidences else 0.0, 3)
        row["Fallback_Rate"]          = round(fallback_count / len(PARAMETERS), 3)
        logger.info(
            f"    {brand} | access={row['Access_Score']} | src_conf={row['Source_Confidence']} "
            f"| fallback_rate={row['Fallback_Rate']}"
        )
        rows.append(row)

    return rows

def _append_rows_to_csv(rows: list[dict]) -> None:
    if not rows:
        return
    df_part = pd.DataFrame(rows)
    for col in SUBMISSION_COLUMN_ORDER:
        if col not in df_part.columns:
            df_part[col] = "Not specified"
    df_part = df_part[SUBMISSION_COLUMN_ORDER]
    write_header = not OUTPUT_CSV.exists()
    df_part.to_csv(OUTPUT_CSV, mode="a", header=write_header, index=False)

def _load_done_filenames() -> set[str]:
    if not OUTPUT_CSV.exists():
        return set()
    try:
        existing = pd.read_csv(OUTPUT_CSV)
        return set(existing["Filename"].astype(str).unique())
    except Exception:
        return set()

def run_pipeline(max_pdfs: int = MAX_PDFS, resume: bool = True) -> pd.DataFrame:
    pdf_paths = sorted(PDF_DIR.glob("*.pdf"))[:max_pdfs]
    if not pdf_paths:
        raise FileNotFoundError(f"No PDFs in {PDF_DIR}")

    done = _load_done_filenames() if resume else set()
    pending = [p for p in pdf_paths if p.name not in done]
    logger.info(f"PDFs total={len(pdf_paths)} done={len(done)} pending={len(pending)}")

    for pdf_path in tqdm(pending, desc="PDFs"):
        try:
            rows = process_pdf(pdf_path)
            _append_rows_to_csv(rows)
            logger.info(f"  ✓ {pdf_path.name}: appended {len(rows)} rows")
        except Exception as e:
            logger.exception(f"FAILED {pdf_path.name}")
            logger.error(f"  ✗ {pdf_path.name}: {e}")

    if not OUTPUT_CSV.exists():
        raise RuntimeError("No rows produced; check OpenRouter API key and PDFs.")

    df = pd.read_csv(OUTPUT_CSV)
    logger.info(f"result.csv has {len(df)} rows ({df['Filename'].nunique()} unique PDFs)")
    return df

Execute the pipeline and display the result rows.

In [ ]:
df = run_pipeline(max_pdfs=MAX_PDFS)

In [ ]:
df.head(5)